In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp

In [0]:

CATALOG = spark.conf.get("catalog")
SILVER_SCHEMA = spark.conf.get("silver_schema")
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_netflix"

GOLD_SCHEMA = spark.conf.get("gold_schema")

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_type")
def gold_titles_by_type():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("type")
        .agg(F.count("*").alias("total_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_audience")
def gold_titles_by_audience():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("audience_category")
        .agg(F.count("*").alias("title_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_release_period")
def gold_titles_by_release_period():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("release_period")
        .agg(F.count("*").alias("title_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_release_period")
def gold_titles_by_release_period():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("release_period")
        .agg(F.count("*").alias("title_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_content")
def gold_content():
    df = spark.read.table(SILVER_TABLE)
    total_titles = df.count()
    return(
        df.groupBy(
            "type", "audience_category", "release_period"
        ).agg(
            F.count("*").alias("title_count")
        ).withColumn(
            "catalog_share", 
            F.round("title_count"/F.lit(total_titles)*100, 2)
        )
    ).orderBy(F.desc("title_count"))